# Recurrent Neural Network

Nessa aula prática, vamos comparar RNNs simples, LSTM e GRU e demonstrar na prática como cada arquitetura lida com o conceito de "memória" ao prever os próximos passos de uma sequência.

Primeiro, vamos importar todas as bibliotecas necessárias e configurar o dispositivo (CPU ou GPU, se disponível).

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

# Configurar o dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

In [ ]:
# Diagnóstico do ambiente — rode esta célula primeiro
import sys, os, sklearn

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("scikit-learn:", sklearn.__version__)
print("CUDA disp.  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
else:
    print(">>> Sem GPU. Esta aula roda na CPU sem problema: as redes são pequenas.")

# Os CSVs desta aula moram no repositório. Rodando localmente, lemos da pasta;
# no Colab (que carrega só o .ipynb, sem clonar o repo) lemos direto do GitHub.
DADOS = "dados" if os.path.exists(os.path.join("dados", "nvda.csv")) else \
    "https://raw.githubusercontent.com/Erickslb/deep-learning-fgv-2026/main/aulas_praticas/dados"
print("Dados       :", DADOS)

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

Vamos começar com um exemplo básico - uma a onda senoidal.

In [ ]:
# 1. Gerar a série temporal (onda senoidal)
amplitude = 1
frequency = 0.1
x = np.linspace(0, 300, 1000)
serie_temporal_np = amplitude * np.sin(frequency * x)

plt.figure(figsize=(10, 4))
plt.plot(x, serie_temporal_np)
plt.xlabel('x')
plt.ylabel('Amplitude')
plt.title('Onda Senoidal')
plt.show()

# A nossa senoide já está nesse intervalo, mas vamos deixar de referencia
scaler = MinMaxScaler(feature_range=(0, 1))
serie_normalizada = scaler.fit_transform(serie_temporal_np.reshape(-1, 1))

# 3. Criar sequências de input/output
def criar_sequencias(dados, tamanho_janela):
    X, y = [], []
    for i in range(len(dados) - tamanho_janela):
        X.append(dados[i:(i + tamanho_janela)])
        y.append(dados[i + tamanho_janela])
    return np.array(X), np.array(y)

tamanho_janela = 50 # Usar 50 passos anteriores para prever o próximo
X, y = criar_sequencias(serie_normalizada, tamanho_janela)

# 4. Dividir em treino e teste (80% treino, 20% teste)
tamanho_treino = int(len(X) * 0.8)
X_treino, X_teste = X[:tamanho_treino], X[tamanho_treino:]
y_treino, y_teste = y[:tamanho_treino], y[tamanho_treino:]

# 5. Converter para tensores do PyTorch e mover para o dispositivo
X_treino = torch.from_numpy(X_treino).float()
y_treino = torch.from_numpy(y_treino).float()
X_teste = torch.from_numpy(X_teste).float()
y_teste = torch.from_numpy(y_teste).float()

print(f"Shape X_treino: {X_treino.shape}") # (amostras, tamanho_janela, features)
print(f"Shape y_treino: {y_treino.shape}")

> ⚠️ **Duas coisas na célula acima que valem atenção — e que você não deve copiar para um trabalho sério:**
>
> 1. **O `scaler` foi ajustado na série inteira antes de separar treino e teste.** Isso é *vazamento de dados* (`data leakage`): o mínimo e o máximo usados na normalização carregam informação do futuro. Numa senoide perfeita o efeito é nulo, mas em série de preços é o bastante para inflar o resultado. O certo é `fit` só no treino e `transform` nos dois.
> 2. **A divisão treino/teste é cronológica, não aleatória** — e isso está certo. Em série temporal, embaralhar antes de dividir deixaria o modelo treinar com dados posteriores aos de teste.
>
> Note também que `shuffle=True` no `DataLoader` (mais abaixo) **não** é o mesmo problema: ali cada amostra já é uma janela fechada, e embaralhar a ordem das janelas dentro do treino é inofensivo.


Agora vamos definir os modelos usando as implementações do Pytorch:

In [ ]:
class ModeloRecorrente(nn.Module):
    def __init__(self, tipo_modelo, input_size, hidden_size, num_layers, output_size, dropout=0.0):
        super(ModeloRecorrente, self).__init__()
        # Nota: nn.RNN/LSTM/GRU só aplicam dropout ENTRE camadas empilhadas.
        # Com num_layers=1 o argumento é ignorado (o PyTorch emite um UserWarning).
        # Se você quer dropout com uma única camada, aplique-o antes do self.fc.
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        if tipo_modelo == 'RNN':
            self.recorrente = nn.RNN(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        elif tipo_modelo == 'LSTM':
            self.recorrente = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        elif tipo_modelo == 'GRU':
            self.recorrente = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # A saída da camada recorrente contém (output, hidden_state)
        # Queremos apenas a saída do último passo de tempo
        out_recorrente, _ = self.recorrente(x)
        out = self.fc(out_recorrente[:, -1, :]) # Usar apenas a última saída da sequência
        return out

A função de treinamento:

In [ ]:
def plot_loss(train_losses):
    plt.figure(figsize=(6, 2))
    plt.plot(train_losses, label='Loss de Treinamento')
    plt.xlabel('Épocas')
    plt.ylabel('Loss')
    plt.title('Loss de Treinamento ao Longo das Épocas')
    plt.legend()
    plt.show()
    
def treinar_modelo(modelo, train_loader, num_epochs, learning_rate=1e-3,
                   print_loss=True, plot=True):
    criterion = nn.MSELoss(reduction='mean') # Erro Quadrático Médio, bom para regressão
    optimizer = torch.optim.Adam(modelo.parameters(), lr=learning_rate)
    train_avg_loss = []

    for epoch in range(num_epochs):
        modelo.train()
        total_loss = 0
        # Cuidado com o nome das variáveis do laço: se elas se chamassem
        # X_treino/y_treino, sobrescreveriam as globais que guardam o conjunto de
        # treino inteiro, deixando lá apenas o último batch. Um
        # `TensorDataset(X_treino, y_treino)` criado depois disso treinaria com 32
        # exemplos em vez de 760 — sem erro e sem aviso.
        for Xb, yb in train_loader:
            Xb = Xb.to(device)
            yb = yb.to(device)

            # Forward pass
            outputs = modelo(Xb)
            loss = criterion(outputs, yb)
            total_loss += loss.item()

            # Backward e otimização
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        
        avg_loss = total_loss / len(train_loader)
        train_avg_loss.append(avg_loss)

        if ((epoch+1) % 10 == 0) and print_loss:
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.5f}')

    if plot:
        plot_loss(train_avg_loss)

    return modelo

E a função para inferência:

In [ ]:
def prever_e_plotar(modelo, X_teste, y_teste, scaler, passos_futuros=None, plot=True):
    # Por padrão prevê exatamente o tamanho do conjunto de teste, para que as duas
    # curvas do gráfico cubram o mesmo intervalo no eixo x.
    if passos_futuros is None:
        passos_futuros = len(y_teste)
    modelo.eval()
    previsoes = []
    input_atual = X_teste[0].unsqueeze(0).clone().detach()
    out = y_teste.shape[1] 

    with torch.no_grad():
        for _ in range(passos_futuros):
            previsao = modelo(input_atual.to(device))
            previsoes.append(previsao.item())
            
            # Atualiza a janela de input: remove o valor mais antigo e adiciona a nova previsão
            nova_sequencia = torch.cat((input_atual[:, 1:, :], previsao.view(1, 1, out).to(input_atual.device)), dim=1)
            input_atual = nova_sequencia[0].unsqueeze(0)

    previsoes_desnormalizadas = scaler.inverse_transform(np.array(previsoes).reshape(-1, 1))
    dados_reais = scaler.inverse_transform(y_teste.cpu().numpy())

    # Plotar
    if plot:
        plt.figure(figsize=(6, 2))
        plt.plot(np.arange(len(dados_reais)), dados_reais, label='Dados Reais (Teste)')
        plt.plot(np.arange(len(previsoes_desnormalizadas)), previsoes_desnormalizadas, label='Previsão do Modelo')
        plt.title(f'Previsão vs Real - {modelo.recorrente.__class__.__name__}')
        plt.xlabel('Passos de Tempo')
        plt.ylabel('Amplitude')
        plt.legend()
        plt.show()
    
    return previsoes_desnormalizadas

> ⏱️ A célula abaixo treina **três** redes por `EPOCHS` épocas cada. Com os valores atuais (100 épocas, 760 amostras, batch 32) dá algo em torno de **1 a 2 minutos na CPU** para as três — as redes são minúsculas. Guarde essa referência: o exercício sugere 400 épocas, o que multiplica esse tempo por 4 **por modelo testado**.


In [ ]:
# Parâmetros dos modelos 
INPUT_SIZE = 1
HIDDEN_SIZE = 64
NUM_LAYERS = 1
OUTPUT_SIZE = 1
EPOCHS = 100
BATCH_SIZE = 32

train_dataset = TensorDataset(X_treino, y_treino)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# --- Modelo 1: RNN Clássica ---
print("\n--- Treinando RNN Clássica ---")
rnn_model = ModeloRecorrente('RNN', INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, OUTPUT_SIZE).to(device)
rnn_model = treinar_modelo(rnn_model, train_loader, num_epochs=EPOCHS)
previsoes_rnn = prever_e_plotar(rnn_model, X_teste, y_teste, scaler)

# # --- Modelo 2: LSTM ---
print("\n--- Treinando LSTM ---")
lstm_model = ModeloRecorrente('LSTM', INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, OUTPUT_SIZE).to(device)
lstm_model = treinar_modelo(lstm_model, train_loader, num_epochs=EPOCHS)
previsoes_lstm = prever_e_plotar(lstm_model, X_teste, y_teste, scaler)

# --- Modelo 3: GRU ---
print("\n--- Treinando GRU ---")
gru_model = ModeloRecorrente('GRU', INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, OUTPUT_SIZE).to(device)
gru_model = treinar_modelo(gru_model, train_loader, num_epochs=EPOCHS)
previsoes_gru = prever_e_plotar(gru_model, X_teste, y_teste, scaler)

Comparando as 3:

In [ ]:
dados_verdadeiros =  scaler.inverse_transform(y_teste.cpu().numpy())

plt.figure(figsize=(6, 2))
plt.plot(np.arange(len(dados_verdadeiros)), dados_verdadeiros, label='Dados Reais (Teste)')
plt.plot(np.arange(len(previsoes_gru)), previsoes_gru, label='GRU')
plt.plot(np.arange(len(previsoes_lstm)), previsoes_lstm, label='LSTM')
plt.plot(np.arange(len(previsoes_rnn)), previsoes_rnn, label='RNN')
plt.title(f'Previsão vs Real ')
plt.xlabel('Passos de Tempo')
plt.ylabel('Amplitude')
plt.legend()
plt.show()

**Reflita por um momento sobre os resultados obtidos nesse exemplo extremamente simples.**

Agora, vamos fazer as mesmas comparações para um dataset mais complexo. Temos duas opções:


**1.  Série de preço/volume das ações da NVIDIA**

A API do yahoo finance retorna as colunas:
- **Open**: O preço da ação no início do dia de negociação (primeiro preço registrado no dia).
- **Close**: O preço da ação no final do dia de negociação (último preço registrado).
- **High**: O preço mais alto atingido pela ação durante o dia de negociação.
- **Low**: O preço mais baixo da ação durante o dia de negociação.
- **Volume**: O número de ações negociadas durante o dia, representado em valores inteiros.

In [ ]:
#script para baixar os dados da NVDA
# !pip install yfinance
# import yfinance as yf
# from datetime import date
# end_date = date.today().strftime("%Y-%m-%d")
# start_date = '1990-01-01'
# df = yf.download('NVDA', start=start_date, end=end_date).droplevel(1, axis=1) 

nvda = pd.read_csv(f'{DADOS}/nvda.csv', parse_dates=['Date'], index_col='Date')
print(nvda.head())

fig, axs = plt.subplots(2, 2, figsize=(12, 4))

axs[0,0].plot(nvda.drop("Volume", axis=1)[:]) # Últimos 60 dias
axs[0,0].set_title('Dados Completos de Preço')

axs[0,1].plot(nvda.drop("Volume", axis=1)[-60:]) # Últimos 60 dias
axs[0,1].set_title('Dados Recentes de Preço')

axs[1,0].plot(nvda["Volume"][:]) # Últimos 60 dias
axs[1,0].set_title('Dados Completos de Volume')

axs[1,1].plot(nvda["Volume"][-60:]) # Últimos 60 dias
axs[1,1].set_title('Dados Recentes de Volume')

plt.tight_layout()
plt.show()

**2. Dados de clima diários de Delhi, India**

O [dataset do kaggle](https://www.kaggle.com/datasets/sumanthvrao/daily-climate-time-series-data/data) possui:

1. meantemp: Mean temperature averaged out from multiple 3 hour intervals in a day.
2. humidity: Humidity value for the day (units are grams of water vapor per cubic meter volume of air).
3. wind_speed: Wind speed measured in kmph.
4. meanpressure: Pressure reading of weather (measure in atm)

In [ ]:
clima = pd.read_csv(f'{DADOS}/Clima_Delhi.csv', parse_dates=['date'], index_col='date')
print(clima.head())

fig, axs = plt.subplots(2, 2, figsize=(12, 4))

axs[0,0].plot(clima['meantemp']) # Últimos 60 dias
axs[0,0].set_title('Temperatrura Média')

axs[0,1].plot(clima["humidity"]) # Últimos 60 dias
axs[0,1].set_title('Umidade')

axs[1,0].plot(clima["wind_speed"]) # Últimos 60 dias
axs[1,0].set_title('Velocidade do Vento')

axs[1,1].plot(clima["meanpressure"].clip(950,1500)) # Últimos 60 dias
axs[1,1].set_title('Pressão Média')

plt.tight_layout()
plt.show()

## Exercícios


1. Varie os parâmetros da rede criada (hidden state, número layers, dropout) para prever a série de preços/clima.
    - Você pode comparar as variações por arquitetura e depois o melhor de cada uma ou comparar as redes com os mesmos parâmetros, por exemplo. 
2. Escolha uma das variações acima e mude o input: preveja preço e volume / temperatura e umidade ao mesmo tempo;
3. Reflita sobre os todos resultados obtidos. Escolha pelo menos gráfico de cada item e poste no [issue da aula](https://github.com/Erickslb/deep-learning-fgv-2026/issues). Faça um breve comentário com suas observações. 
- [Opcional] Implemente a variação CNN+LSTM. (São poucas linhas de código a mais)
- [Opcional] Você pode comparar seu resultados com modelos de séries temporais clássicos como ARIMA, ARMA, MA. .

Dicas/Notas:
- As redes treinam muito rápido e poucas alterações precisam ser feitas no código;
- Você não precisa exagerar nos testes (só se quiser), mas aumente um pouco o tamanho da rede para esses datasets para ver a diferença;
Exemplos: (HD: [64, 128]; NL: [1,3]; Dropout:[0, 0.2] já são 8 modelos por arquitetura.);
- Treine por mais de 100 epochs (sugestão: ~400);
- Caso demore mais de 7 minutos em um dos passos, veja a resposta abaixo para continuar. 

> 💡 A resolução deste exercício está no notebook correspondente da pasta `solucoes/`. Tente resolver por conta antes de consultar — o erro que você comete sozinho é o que você não repete depois.


## Avaliação

Deixe seu feedback da aula pelo **formulário linkado na coluna _Feedback_ do
[README do repositório](https://github.com/Erickslb/deep-learning-fgv-2026)** — o que funcionou, o que ficou confuso, o que faltou.

Dúvidas técnicas que podem interessar aos colegas ficam melhor como
[issue](https://github.com/Erickslb/deep-learning-fgv-2026/issues), que todo mundo vê.